# CardioIA — Fase 2 — Parte 1
## Diagnóstico automatizado a partir de frases de sintomas

Este notebook lê frases de pacientes descrevendo sintomas (`data/frases_sintomas.txt`), usa um **mapa de conhecimento** simples (`data/mapa_conhecimento.csv`) que associa expressões de sintomas a possíveis doenças, e sugere um diagnóstico para cada frase.

A técnica usada é bem simples de propósito: normalização de texto (minúsculas + remoção de acentos) e **busca de substring** das expressões do mapa de conhecimento dentro de cada frase. Não é NLP sofisticado — é justamente o tipo de regra simples que a atividade pede, para simular como um sistema de apoio ao diagnóstico organiza conhecimento estruturado.

**Sobre a fidelidade clínica:** cada linha do mapa de conhecimento foi checada contra uma fonte médica reconhecida (Manuais MSD, diretrizes da Sociedade Brasileira de Cardiologia, Ministério da Saúde) — a coluna `fonte` do CSV registra de onde veio cada associação sintoma → doença. A lista completa de referências está na Seção 5, ao final deste notebook.

In [1]:
import csv
import unicodedata
from collections import Counter
import pandas as pd

CAMINHO_FRASES = "../data/frases_sintomas.txt"
CAMINHO_MAPA = "../data/mapa_conhecimento.csv"

## 1. Funções auxiliares

Normalização de texto (remove acentuação e caixa) para deixar a comparação de substrings mais robusta a pequenas variações de escrita.

In [2]:
def normalizar(texto: str) -> str:
    """Deixa o texto em minúsculas e remove acentuação, para facilitar a comparação de substrings."""
    texto = texto.lower().strip()
    texto = unicodedata.normalize("NFKD", texto)
    return "".join(c for c in texto if not unicodedata.combining(c))


def carregar_mapa_conhecimento(caminho_csv: str):
    """Lê o CSV sintoma_1 | sintoma_2 | doenca_associada | fonte e retorna uma lista de
    tuplas (sintoma_normalizado, sintoma_original, doenca), uma por coluna de sintoma."""
    mapa = []
    with open(caminho_csv, encoding="utf-8") as f:
        leitor = csv.DictReader(f)
        for linha in leitor:
            for coluna in ("sintoma_1", "sintoma_2"):
                sintoma = linha[coluna].strip()
                if sintoma:
                    mapa.append((normalizar(sintoma), sintoma, linha["doenca_associada"].strip()))
    return mapa


def diagnosticar_frase(frase: str, mapa):
    """Procura, dentro da frase normalizada, todas as expressões de sintoma do mapa de
    conhecimento. Cada doença recebe uma pontuação igual à soma do tamanho dos sintomas
    encontrados que apontam para ela (expressões mais longas/específicas pesam mais).
    Retorna a lista de sintomas encontrados e a doença com maior pontuação."""
    frase_norm = normalizar(frase)
    encontrados = {}
    for sintoma_norm, sintoma_original, doenca in mapa:
        if sintoma_norm in frase_norm and sintoma_norm not in encontrados:
            encontrados[sintoma_norm] = (sintoma_original, doenca)

    if not encontrados:
        return [], None

    pontuacao = Counter()
    for sintoma_norm, (_, doenca) in encontrados.items():
        pontuacao[doenca] += len(sintoma_norm)

    doenca_sugerida = pontuacao.most_common(1)[0][0]
    sintomas_encontrados = [sintoma_original for sintoma_original, _ in encontrados.values()]
    return sintomas_encontrados, doenca_sugerida

## 2. Carregando o mapa de conhecimento

Cada linha traz também a fonte clínica que embasa aquela associação sintoma → doença.

In [3]:
mapa_conhecimento = carregar_mapa_conhecimento(CAMINHO_MAPA)
df_mapa = pd.read_csv(CAMINHO_MAPA)
print(f"{len(df_mapa)} linhas no mapa de conhecimento, cobrindo {df_mapa['doenca_associada'].nunique()} doenças diferentes.")
df_mapa[["sintoma_1", "sintoma_2", "doenca_associada"]]

15 linhas no mapa de conhecimento, cobrindo 9 doenças diferentes.


,sintoma_1,sintoma_2,doenca_associada
0,dor no peito,aperto no torax,Infarto Agudo do Miocardio
1,suor frio,nausea,Infarto Agudo do Miocardio
2,esforco fisico,melhora com o repouso,Angina
3,cansaco constante,fadiga,Insuficiencia Cardiaca
4,falta de ar,dificuldade para respirar,Insuficiencia Cardiaca
5,inchaco nas pernas,inchaco nos pes,Insuficiencia Cardiaca
6,pernas inchadas,pes inchados,Insuficiencia Cardiaca
7,tosse persistente,falta de ar ao se deitar,Insuficiencia Cardiaca
8,palpitacao,coracao acelerado,Arritmia Cardiaca
9,batimento irregular,pulso irregular,Fibrilacao Atrial


## 3. Lendo as frases de sintomas dos pacientes

In [4]:
with open(CAMINHO_FRASES, encoding="utf-8") as f:
    frases = [linha.strip() for linha in f if linha.strip()]

print(f"{len(frases)} frases carregadas.\n")
for i, frase in enumerate(frases, 1):
    print(f"{i:>2}. {frase}")

10 frases carregadas.

 1. Há dois dias estou com uma dor no peito que piora quando faço esforço físico, e isso já me impede de subir escadas no trabalho.
 2. Sinto cansaço constante há uma semana, mesmo depois de descansar, e não consigo mais terminar minhas tarefas domésticas.
 3. Desde ontem à noite tenho falta de ar mesmo em repouso, o que está me deixando com medo de dormir sozinho.
 4. Há três dias percebi inchaço nas pernas, principalmente no fim da tarde, e minhas calças já não fecham mais.
 5. Há cerca de um mês percebo palpitação e o coração acelerado sempre que subo uma ladeira, e preciso parar no meio do caminho.
 6. Ontem senti uma dor no peito em aperto acompanhada de suor frio e náusea, e precisei parar de trabalhar imediatamente.
 7. Há dois meses tenho dor nas pernas ao caminhar, que melhora quando paro, e isso já reduziu bastante minhas caminhadas diárias.
 8. Esta manhã percebi fraqueza de um lado do corpo e dificuldade para falar, o que assustou muito minha família 

## 4. Extraindo sintomas e sugerindo diagnóstico para cada frase

In [5]:
resultados = []
for frase in frases:
    sintomas, doenca = diagnosticar_frase(frase, mapa_conhecimento)
    resultados.append({
        "frase": frase,
        "sintomas_identificados": ", ".join(sintomas) if sintomas else "(nenhum sintoma reconhecido)",
        "diagnostico_sugerido": doenca if doenca else "Não foi possível sugerir diagnóstico",
    })

df_resultados = pd.DataFrame(resultados)
pd.set_option("display.max_colwidth", None)
df_resultados

,frase,sintomas_identificados,diagnostico_sugerido
0,"Há dois dias estou com uma dor no peito que piora quando faço esforço físico, e isso já me impede de subir escadas no trabalho.","dor no peito, esforco fisico",Angina
1,"Sinto cansaço constante há uma semana, mesmo depois de descansar, e não consigo mais terminar minhas tarefas domésticas.",cansaco constante,Insuficiencia Cardiaca
2,"Desde ontem à noite tenho falta de ar mesmo em repouso, o que está me deixando com medo de dormir sozinho.",falta de ar,Insuficiencia Cardiaca
3,"Há três dias percebi inchaço nas pernas, principalmente no fim da tarde, e minhas calças já não fecham mais.",inchaco nas pernas,Insuficiencia Cardiaca
4,"Há cerca de um mês percebo palpitação e o coração acelerado sempre que subo uma ladeira, e preciso parar no meio do caminho.","palpitacao, coracao acelerado",Arritmia Cardiaca
5,"Ontem senti uma dor no peito em aperto acompanhada de suor frio e náusea, e precisei parar de trabalhar imediatamente.","dor no peito, suor frio, nausea",Infarto Agudo do Miocardio
6,"Há dois meses tenho dor nas pernas ao caminhar, que melhora quando paro, e isso já reduziu bastante minhas caminhadas diárias.",dor nas pernas ao caminhar,Doenca Arterial Periferica
7,"Esta manhã percebi fraqueza de um lado do corpo e dificuldade para falar, o que assustou muito minha família durante o café da manhã.","fraqueza de um lado do corpo, dificuldade para falar",Acidente Vascular Cerebral (AVC)
8,"Há uma semana tenho batimento irregular e sinto o pulso irregular, principalmente à noite, o que tem atrapalhado meu sono.","batimento irregular, pulso irregular",Fibrilacao Atrial
9,"Há três dias sinto dor de cabeça forte junto com visão embaçada, e isso tem dificultado meu trabalho no computador.","dor de cabeca forte, visao embacada",Crise Hipertensiva


In [6]:
for i, linha in df_resultados.iterrows():
    print(f"Paciente {i + 1}")
    print(f"  Frase: {linha['frase']}")
    print(f"  Sintomas identificados: {linha['sintomas_identificados']}")
    print(f"  Diagnóstico sugerido: {linha['diagnostico_sugerido']}")
    print()

Paciente 1
  Frase: Há dois dias estou com uma dor no peito que piora quando faço esforço físico, e isso já me impede de subir escadas no trabalho.
  Sintomas identificados: dor no peito, esforco fisico
  Diagnóstico sugerido: Angina

Paciente 2
  Frase: Sinto cansaço constante há uma semana, mesmo depois de descansar, e não consigo mais terminar minhas tarefas domésticas.
  Sintomas identificados: cansaco constante
  Diagnóstico sugerido: Insuficiencia Cardiaca

Paciente 3
  Frase: Desde ontem à noite tenho falta de ar mesmo em repouso, o que está me deixando com medo de dormir sozinho.
  Sintomas identificados: falta de ar
  Diagnóstico sugerido: Insuficiencia Cardiaca

Paciente 4
  Frase: Há três dias percebi inchaço nas pernas, principalmente no fim da tarde, e minhas calças já não fecham mais.
  Sintomas identificados: inchaco nas pernas
  Diagnóstico sugerido: Insuficiencia Cardiaca

Paciente 5
  Frase: Há cerca de um mês percebo palpitação e o coração acelerado sempre que subo u

## 5. Fontes consultadas para validar o mapa de conhecimento

Este protótipo tem como referência sites e diretrizes clínicas oficias para se basear nos sintomas e suas associações.

### Referências

- Manual MSD (edição profissional). *Infarto Agudo do Miocárdio (IAM)*. https://www.msdmanuals.com/pt/profissional/doencas-cardiovasculares/doenca-coronariana/infarto-agudo-do-miocardio-iam
- Manual MSD (versão para a família). *Angina*. https://www.msdmanuals.com/pt/casa/disturbios-do-coracao-e-dos-vasos-sanguineos/doenca-arterial-coronariana/angina
- Sociedade Brasileira de Cardiologia. *II Diretriz Brasileira de Insuficiência Cardíaca Aguda*. https://www.scielo.br/j/abc/a/6CWscRNFQdbmnHBVMqjBtfC/?format=html&lang=pt
- Manual MSD (versão para a família). *Considerações gerais sobre arritmias cardíacas*. https://www.msdmanuals.com/pt/casa/disturbios-do-coracao-e-dos-vasos-sanguineos/arritmias-cardiacas/consideracoes-gerais-sobre-arritmias-cardiacas
- Manual MSD (edição profissional). *Fibrilação Atrial*. https://www.msdmanuals.com/pt/profissional/doencas-cardiovasculares/arritmias-cardiacas-especificas/fibrilacao-atrial
- Manual MSD (edição profissional). *Síncope*. https://www.msdmanuals.com/pt/profissional/doencas-cardiovasculares/sintomas-de-doencas-cardiovasculares/sincope
- Sociedade Brasileira de Cardiologia. *Posicionamento Luso-Brasileiro de Emergências Hipertensivas – 2020*. https://www.ncbi.nlm.nih.gov/pmc/articles/PMC9744343/
- Sociedade Brasileira de Cardiologia. *Diretrizes Brasileiras de Hipertensão Arterial – 2020*. https://pmc.ncbi.nlm.nih.gov/articles/PMC9949730/
- Ministério da Saúde. *AVC*. https://www.gov.br/saude/pt-br/assuntos/saude-de-a-a-z/a/avc
- Manual MSD (edição profissional). *Doença Arterial Periférica*. https://www.msdmanuals.com/pt-pt/profissional/doencas-cardiovasculares/disturbios-arteriais-perifericos/doenca-arterial-periferica

## 6. Limitações que permanecem, mesmo com fontes citadas

Mesmo depois desta checagem, o protótipo continua sendo uma simplificação, e vale reforçar por quê:

- **Correspondência literal (substring), não semântica:** o algoritmo só reconhece um sintoma se a expressão (ou uma bem próxima) aparecer na frase. Sinônimos não previstos, erros de digitação ou expressões regionais não são capturados. Um sistema real usaria técnicas de NLP mais robustas (lematização, embeddings, reconhecimento de entidades).
- **Não entende negação nem contexto temporal:** "não sinto mais dor no peito" seria tratado como "sinto dor no peito".
- **Sintomas compartilhados por várias doenças:** mesmo com fontes, muitos sintomas (dor no peito, falta de ar) aparecem em múltiplas condições — cardíacas e não cardíacas. O critério de desempate usado aqui (pontuação por especificidade da expressão) é uma escolha de engenharia, não uma regra diagnóstica clínica.
- **Cobertura ainda pequena:** apenas 10 doenças estão no mapa; diagnósticos diferenciais não cardíacos (ansiedade, refluxo, problemas musculoesqueléticos) não entram na disputa, o que infla artificialmente a confiança nos diagnósticos cardíacos sugeridos.
- **Nenhuma validação clínica formal:** as fontes usadas são confiáveis e de acesso público (Manual MSD, diretrizes da SBC, Ministério da Saúde), mas a curadoria de qual sintoma vai em qual linha do CSV ainda foi feita por uma única pessoa não-médica. Um sistema real precisaria de revisão por profissionais de saúde antes de qualquer uso além do didático.
- **Este protótipo não deve ser usado para diagnóstico real** — é um exercício para entender, na prática, como sistemas de IA em saúde estruturam conhecimento e cruzam texto livre com uma base de conhecimento, agora com o cuidado adicional de checar essa base contra a literatura.